# 🚀 Number Guessing Game

**Lumexa Python Foundations · Course 1: Python Programming Essentials · Lesson 5**

| | |
|---|---|
| **Project** | Number Guessing Game (portfolio project 2 of 9) |
| **You will practise** | `random`, `while` loops, counters, `if` / `elif` / `else`, scores |
| **Time** | about 45 minutes |

Build a game where the computer thinks of a secret number and you have a limited number of attempts to find it. You will use loops, random numbers, hints and score tracking, and at the end you will see why **seven attempts is always enough** for the numbers 1 to 100.

## 🧭 How to use this notebook

1. **Keep your own copy.** Choose **File → Save a copy in Drive** first, so your work is saved in *your* Google Drive.
2. **Run the cells in order**, from the top, with **Shift + Enter** (or **Runtime → Run all**). Read the text above each cell before you run it.
3. Cells marked **✏️ Your turn** are for you to change or complete. They are safe to run before you finish them.
4. If something goes wrong, use **Runtime → Restart session and run all**. A notebook you can run from the top is a notebook you can trust.
5. **Demo mode:** by default the notebook "types" the answers for you, so *Run all* works without stopping. To play yourself, set `INTERACTIVE = True` in the setup cell and run it again.

In [ ]:
# ⚙️ Setup — run this cell first
# INTERACTIVE = False  -> the notebook answers for you ("demo mode"), so Run all works.
# INTERACTIVE = True   -> you type the answers yourself.
INTERACTIVE = False

def ask(prompt, demo_answer=""):
    """Works like input(). In demo mode it 'types' demo_answer for you."""
    if INTERACTIVE:
        return input(prompt)
    print(f"{prompt}{demo_answer}")
    return str(demo_answer)

print("Setup complete. Interactive mode is", "ON" if INTERACTIVE else "OFF (demo mode)")

import random

if not INTERACTIVE:
    random.seed(2030)     # demo mode: the "random" number repeats, so everyone sees the same game

## Part 1 · A secret number and one guess

`random.randint(1, 100)` picks a whole number from 1 to 100, **both included**. We compare the guess with the secret using `if`, `elif` and `else`.

In [ ]:
secret = random.randint(1, 100)
guess = int(ask("Guess a number from 1 to 100: ", 50))

if guess == secret:
    print("🎉 Correct!")
elif guess < secret:
    print("Too low!")
else:
    print("Too high!")

## Part 2 · A loop with limited attempts

One guess is not a game. A `while` loop keeps asking **while there are attempts left**. We use a **counter** (`attempts_left`) that goes down by one each time, and `break` to stop early when the player wins.

Look carefully at the order inside the loop: ask → count the attempt → compare → give a hint.

In [ ]:
secret = random.randint(1, 100)
attempts_left = 7
won = False

while attempts_left > 0:
    guess = int(ask(f"[{attempts_left} attempts left] Your guess: ", 50))
    attempts_left -= 1
    if guess == secret:
        won = True
        break
    elif guess < secret:
        print("   Too low!")
    else:
        print("   Too high!")

print("You won! 🎉" if won else f"Out of attempts. The number was {secret}.")

## Part 3 · The complete round: hints, validation and a score

Now we put one whole round into a **function** so we can play it again and again (you will study functions in Lesson 8, but you can already read this one).

New ideas in this version:

- We **remember the range** (`low` to `high`) that is still possible, and show it as a hint.
- The **score** rewards fast wins: `attempts_left * 10 + 10`.
- `text.strip().isdigit()` checks that the player typed a whole number. (You will learn `.isdigit()` in Lesson 6.) A typing mistake does not cost an attempt.
- In demo mode the "player" always guesses the **middle** of the range. That is a clever strategy, and we will find out why below.

In [ ]:
MAX_ATTEMPTS = 7
LOWEST, HIGHEST = 1, 100

def play_round(round_number=1):
    """Play one round. Returns the score (0 if the player lost)."""
    secret = random.randint(LOWEST, HIGHEST)
    attempts_left = MAX_ATTEMPTS
    low, high = LOWEST, HIGHEST
    print(f"\n🎯 Round {round_number}: I am thinking of a number from {LOWEST} to {HIGHEST}.")

    while attempts_left > 0:
        text = ask(f"[{attempts_left} left] Your guess ({low}-{high}): ", (low + high) // 2)
        if not text.strip().isdigit():
            print("   Please type a whole number.")
            continue                      # a mistake does not cost an attempt
        guess = int(text)
        attempts_left -= 1

        if guess == secret:
            score = attempts_left * 10 + 10
            print(f"🎉 Correct! The number was {secret}. Score: {score}")
            return score
        elif guess < secret:
            print("   Too low! ⬆️")
            low = max(low, guess + 1)
        else:
            print("   Too high! ⬇️")
            high = min(high, guess - 1)

    print(f"💥 Out of attempts. The number was {secret}.")
    return 0

score = play_round()

## Part 4 · Play again and keep the best score

An outer loop asks "Play again?" and a variable remembers the **best score**. `max(a, b)` returns the bigger of two numbers.

In [ ]:
best_score = 0
rounds_played = 0
demo_answers = iter(["yes", "yes", "no"])

while True:
    rounds_played += 1
    score = play_round(rounds_played)
    best_score = max(best_score, score)
    again = ask("\nPlay again? (yes/no): ", next(demo_answers, "no")).strip().lower()
    if again != "yes":
        break

print(f"\nRounds played: {rounds_played}  |  Best score: {best_score}")

## ✏️ Your turn · Change the rules

Make the game your own. Try **one** of these, then run the cell:

- Change `MAX_ATTEMPTS` to `3`. What happens to your chances? (Run Part 3 and Part 4 again.)
- Change `LOWEST, HIGHEST` to `1, 1000`. How many attempts do you need now?
- Add a **"warm / cold" hint**: if `abs(guess - secret) <= 5`, print `"🔥 Very warm!"`.
- Give extra points for a win on the first attempt.

Write what you changed and what you noticed here (double-click): *…*

In [ ]:
# ✏️ Your turn: copy play_round() here and change it, or just change MAX_ATTEMPTS above and re-run.
# Example: a harder game with only 3 attempts
MAX_ATTEMPTS = 3
play_round(99)
MAX_ATTEMPTS = 7      # put the normal rule back

## Part 5 · Why seven attempts is enough (a bonus experiment)

The demo player always guesses the **middle** of the numbers that are still possible, so every guess throws away **half** of them: 100 → 50 → 25 → 13 → 7 → 4 → 2 → 1. That takes at most 7 guesses. Computer scientists call this **binary search**.

Let a robot play **1,000 games** with this strategy and measure it:

In [ ]:
def bot_game(secret):
    """Return how many guesses the middle-of-the-range strategy needs."""
    low, high, attempts = 1, 100, 0
    while True:
        guess = (low + high) // 2
        attempts += 1
        if guess == secret:
            return attempts
        elif guess < secret:
            low = guess + 1
        else:
            high = guess - 1

random.seed(1)
total_attempts = 0
worst_case = 0
for game in range(1000):
    attempts = bot_game(random.randint(1, 100))
    total_attempts += attempts
    worst_case = max(worst_case, attempts)

print(f"Average attempts: {total_attempts / 1000:.2f}")
print(f"Worst case:       {worst_case} attempts")

## 🏆 Stretch challenges

1. Keep a **list of every guess** and print it at the end (Lesson 7).
2. Let the player choose a **difficulty**: easy (1–50, 8 attempts), normal (1–100, 7), hard (1–500, 9).
3. Make the computer guess *your* number. You answer `higher`, `lower` or `correct`.
4. Store the best score in a **file** so it survives between sessions (Course 2).

## 🪞 Reflection

- Which variable is the **counter**? Which line changes it?
- Why does the loop need `break` or a `return`? What would happen without it?
- Why is guessing the middle better than guessing 1, 2, 3, …?

*Your answers:*

## 📤 How to submit

**File → Save a copy in Drive**, then **Share → Anyone with the link can view**, and send the link to your teacher. Check that the notebook runs from the top with **Runtime → Restart session and run all**.